In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.preprocessing import LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

# load data
ratings = pd.read_csv('../data/raw/ratings.csv')
links = pd.read_csv('../data/processed/links_cleaned.csv')
movies = pd.read_csv('../data/processed/movies_final.csv')

print("Ratings shape:", ratings.shape)
print("Unique users:", ratings['userId'].nunique())
print("Unique movies:", ratings['movieId'].nunique())

Ratings shape: (25000095, 4)
Unique users: 162541
Unique movies: 59047


In [5]:
# only keep users who rated at least 50 movies
user_counts = ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 50].index
ratings_filtered = ratings[ratings['userId'].isin(active_users)]

# only keep movies with at least 100 ratings
movie_counts = ratings_filtered['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 100].index
ratings_filtered = ratings_filtered[ratings_filtered['movieId'].isin(popular_movies)]

print("After filtering:")
print("Ratings:", ratings_filtered.shape)
print("Unique users:", ratings_filtered['userId'].nunique())
print("Unique movies:", ratings_filtered['movieId'].nunique())

# check sparsity
total_possible = ratings_filtered['userId'].nunique() * ratings_filtered['movieId'].nunique()
sparsity = 1 - (len(ratings_filtered) / total_possible)
print(f"Matrix sparsity: {sparsity:.3%}")

After filtering:
Ratings: (22553425, 4)
Unique users: 102492
Unique movies: 10234
Matrix sparsity: 97.850%


In [7]:
# take top 20000 most active users
top_users = user_counts[user_counts >= 50].nlargest(20000).index
ratings_sample = ratings_filtered[ratings_filtered['userId'].isin(top_users)]

# take top 5000 most rated movies
top_movies = movie_counts[movie_counts >= 100].nlargest(5000).index
ratings_sample = ratings_sample[ratings_sample['movieId'].isin(top_movies)]

print("Final sample:")
print("Ratings:", ratings_sample.shape)
print("Unique users:", ratings_sample['userId'].nunique())
print("Unique movies:", ratings_sample['movieId'].nunique())

sparsity = 1 - (len(ratings_sample) / (ratings_sample['userId'].nunique() * ratings_sample['movieId'].nunique()))
print(f"Sparsity: {sparsity:.3%}")

Final sample:
Ratings: (11381066, 4)
Unique users: 20000
Unique movies: 5000
Sparsity: 88.619%


In [9]:
# encode user and movie ids to matrix indices
# raw ids like 12345 cant be used as matrix indices directly
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings_sample['user_idx'] = user_encoder.fit_transform(ratings_sample['userId'])
ratings_sample['movie_idx'] = movie_encoder.fit_transform(ratings_sample['movieId'])

n_users = ratings_sample['user_idx'].nunique()
n_movies = ratings_sample['movie_idx'].nunique()

print(f"Matrix size: {n_users} users × {n_movies} movies")

# build sparse matrix
# csr_matrix = compressed sparse row — memory efficient format
sparse_matrix = csr_matrix(
    (ratings_sample['rating'].values,
     (ratings_sample['user_idx'].values,
      ratings_sample['movie_idx'].values)),
    shape=(n_users, n_movies)
)

print("Sparse matrix shape:", sparse_matrix.shape)
print(f"Memory usage: {sparse_matrix.data.nbytes / 1024 / 1024:.1f} MB")


Matrix size: 20000 users × 5000 movies
Sparse matrix shape: (20000, 5000)
Memory usage: 86.8 MB


In [11]:
# mean center — subtract each user's average rating
# this removes user bias (some users always rate high, some always rate low)
# after centering, a positive value means "liked more than usual"
# and negative means "liked less than usual"

# convert to array for mean calculation
matrix_dense = sparse_matrix.toarray().astype(float)

# calculate mean rating per user (ignoring zeros which mean unrated)
user_ratings_mean = np.true_divide(
    matrix_dense.sum(axis=1),
    (matrix_dense != 0).sum(axis=1)
)

# subtract user mean from each rating (only where rating exists)
matrix_centered = matrix_dense.copy()
for i in range(n_users):
    mask = matrix_dense[i] != 0
    matrix_centered[i][mask] -= user_ratings_mean[i]

print("Mean centering done")
print(f"User 0 mean rating: {user_ratings_mean[0]:.2f}")
print(f"User 0 centered ratings sample: {matrix_centered[0][matrix_centered[0] != 0][:5]}")

Mean centering done
User 0 mean rating: 3.70
User 0 centered ratings sample: [0.30216383 0.80216383 0.80216383 1.30216383 0.30216383]


In [13]:
# k = number of latent factors
# think of these as hidden taste dimensions
# 50 is a good balance between accuracy and speed
k = 50

print(f"Running SVD with k={k} latent factors...")
print("This may take 1-2 minutes...")

# svds returns three matrices:
# U = user-factor matrix (n_users × k)
# sigma = importance of each factor (k,)  
# Vt = movie-factor matrix (k × n_movies)
U, sigma, Vt = svds(matrix_centered, k=k)

# convert sigma from 1D array to diagonal matrix
sigma = np.diag(sigma)

print("SVD complete!")
print(f"U shape (users × factors): {U.shape}")
print(f"Sigma shape (factors × factors): {sigma.shape}")
print(f"Vt shape (factors × movies): {Vt.shape}")

Running SVD with k=50 latent factors...
This may take 1-2 minutes...
SVD complete!
U shape (users × factors): (20000, 50)
Sigma shape (factors × factors): (50, 50)
Vt shape (factors × movies): (50, 5000)


In [15]:
# multiply back to get predicted ratings for every user-movie pair
predicted_ratings = np.dot(np.dot(U, sigma), Vt)

# add back user means to get actual rating scale
predicted_ratings_uncentered = predicted_ratings + user_ratings_mean.reshape(-1, 1)

print("Predicted ratings matrix shape:", predicted_ratings_uncentered.shape)
print(f"Rating range: {predicted_ratings_uncentered.min():.2f} to {predicted_ratings_uncentered.max():.2f}")
print(f"Sample predictions for user 0: {predicted_ratings_uncentered[0][:5]}")

Predicted ratings matrix shape: (20000, 5000)
Rating range: -1.32 to 7.44
Sample predictions for user 0: [4.02349754 3.7418105  3.66042302 3.68903452 3.61587989]


In [17]:
# map movieId back to matrix index
movie_id_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_encoder.classes_)}
idx_to_movie_id = {idx: movie_id for idx, movie_id in enumerate(movie_encoder.classes_)}

# map userId to matrix index
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(user_encoder.classes_)}

def get_collaborative_recommendations(user_id, n=10):
    if user_id not in user_id_to_idx:
        return f"User {user_id} not found — they may not have enough ratings"
    
    user_idx = user_id_to_idx[user_id]
    
    # get this user's predicted ratings for all 5000 movies
    user_predictions = predicted_ratings_uncentered[user_idx]
    
    # get movies this user has ALREADY rated — we don't want to recommend these
    already_rated = set(ratings_sample[ratings_sample['userId'] == user_id]['movieId'].values)
    
    # build results
    recommendations = []
    for movie_idx, pred_rating in enumerate(user_predictions):
        movie_id = idx_to_movie_id[movie_idx]
        if movie_id not in already_rated:
            recommendations.append((movie_id, pred_rating))
    
    # sort by predicted rating
    recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)[:n]
    
    # map to movie titles via links
    result_rows = []
    for movie_id, pred_rating in recommendations:
        tmdb_row = links[links['movieId'] == movie_id]
        if len(tmdb_row) > 0:
            tmdb_id = tmdb_row['tmdbId'].values[0]
            movie_row = movies[movies['id'] == tmdb_id]
            if len(movie_row) > 0:
                result_rows.append({
                    'title': movie_row['title'].values[0],
                    'predicted_rating': round(pred_rating, 2),
                    'actual_avg': round(movie_row['vote_average'].values[0], 2)
                })
    
    return pd.DataFrame(result_rows)

# test with a real user
test_user = ratings_sample['userId'].iloc[0]
print(f"Recommendations for user {test_user}:")
get_collaborative_recommendations(test_user)

Recommendations for user 3:


,title,predicted_rating,actual_avg
0,Spirited Away,4.35,8.3
1,Se7en,4.23,8.1
2,Dr. Strangelove or: How I Learned to Stop Worr...,4.21,8.0
3,Princess Mononoke,4.21,8.2
4,L.A. Confidential,4.19,7.7
5,Amélie,4.14,7.8
6,Moon,4.10,7.6
7,Brazil,4.10,7.5


In [19]:
# clip predictions to valid range
predicted_ratings_uncentered = np.clip(predicted_ratings_uncentered, 1, 5)

# save
with open('../models/svd_predictions.pkl', 'wb') as f:
    pickle.dump(predicted_ratings_uncentered, f)

with open('../models/user_encoder.pkl', 'wb') as f:
    pickle.dump(user_encoder, f)

with open('../models/movie_encoder.pkl', 'wb') as f:
    pickle.dump(movie_encoder, f)

with open('../models/user_id_to_idx.pkl', 'wb') as f:
    pickle.dump(user_id_to_idx, f)

with open('../models/idx_to_movie_id.pkl', 'wb') as f:
    pickle.dump(idx_to_movie_id, f)

# save filtered ratings sample for later use
ratings_sample.to_csv('../data/processed/ratings_sample.csv', index=False)

print("All models saved!")
print(f"Predictions matrix: {predicted_ratings_uncentered.shape}")

All models saved!
Predictions matrix: (20000, 5000)
